# QLoRA Fine-Tuning of Qwen3.5-9B on a Google Colab T4 GPU

This notebook performs **QLoRA** (4-bit quantized LoRA) fine-tuning of
`Qwen/Qwen3.5-9B` on a single T4 GPU (16 GB VRAM), following the v4
adaptive-operator post-training pipeline:

1. **Setup** -- install transformers / peft / trl / bitsandbytes
2. **Mount Drive** -- persistent storage for data, adapters, and merged model
3. **Configuration** -- all hyperparameters as editable constants
4. **Load model** -- 4-bit NF4 quantization (critical for T4)
5. **SFT** -- supervised fine-tuning with the Qwen chat template
6. **DPO** -- direct preference optimization, continuing from the SFT adapter
7. **Merge & save** -- merge LoRA weights into the base model (fp16)
8. **Smoke test** -- load merged model and verify control tokens + tool-call format

### T4 constraints honored throughout
| Constraint | Value | Reason |
|---|---|---|
| `MAX_SEQ_LENGTH` | 2048 | T4 VRAM (not 4096) |
| `BATCH_SIZE` | 1 + grad accum 8 | VRAM |
| `gradient_checkpointing` | True | mandatory on T4 |
| `use_cache` | False (training) | incompatible with grad checkpointing |
| compute dtype | fp16 | bf16 not supported on T4 |
| quantization | 4-bit NF4 + double quant | fits 9B model in ~6 GB |


## Cell 1 -- Setup and Installation

Install the Hugging Face stack needed for QLoRA. Run this once per Colab session.


In [ ]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate torch


## Cell 2 -- Mount Google Drive for persistent storage

Mounting Drive means datasets, LoRA adapters, and the final merged model survive
across sessions. We create a project directory tree under
`/content/drive/MyDrive/adaptive_operator_v4`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Create project directory in Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/adaptive_operator_v4'
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/models', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)


## Cell 3 -- Configuration (editable constants)

All hyperparameters live here so you can tune without hunting through cells.
The values are chosen for a **T4 (16 GB)** constraint.


In [ ]:
# === Configuration ===
BASE_MODEL = "Qwen/Qwen3.5-9B"
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
SFT_EPOCHS = 3
SFT_LEARNING_RATE = 1e-4  # QLoRA needs higher LR than full FT
SFT_BATCH_SIZE = 1
SFT_GRADIENT_ACCUMULATION = 8
MAX_SEQ_LENGTH = 2048  # T4 memory constraint
DPO_EPOCHS = 1
DPO_LEARNING_RATE = 5e-5
DPO_BETA = 0.1
PROJECT_DIR = '/content/drive/MyDrive/adaptive_operator_v4'


## Cell 4 -- Load model in 4-bit quantization

We use **bitsandbytes 4-bit NF4 quantization with double quantization** and an
**fp16 compute dtype** (bf16 is not supported on the T4). This is what lets a
9B-parameter model fit in 16 GB VRAM alongside optimizer state and activations.

We also load the tokenizer and ensure a padding token is set (Qwen uses the EOS
token as pad by default).


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

# 4-bit NF4 quantization config -- mandatory for T4 (16 GB VRAM).
# double_quant adds a second quantization stage to save a bit more memory.
# compute_dtype is fp16 because bf16 is NOT supported on the T4 GPU.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # fp16, not bf16 (T4)
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL} in 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    padding_side="right",
)

# Ensure a pad token exists (Qwen has no default pad token).
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Required for QLoRA: prepare model for k-bit training + gradient checkpointing.
model.config.use_cache = False  # incompatible with gradient checkpointing
model.config.pretraining_tp = 1  # disable tensor-parallel tricks on single GPU

print(f"Model loaded. Device map: {model.hf_device_map if hasattr(model, 'hf_device_map') else 'auto'}")
print(f"Pad token: {tokenizer.pad_token!r} (id {tokenizer.pad_token_id})")


## Cell 5 -- QLoRA SFT Training

Supervised fine-tuning on `sft_train.jsonl`. Each record is a
`{"messages": [...]}` conversation (system / user / assistant) that we format
with the **Qwen chat template**.

Key T4 details:
- LoRA targets all linear projections: `q_proj, k_proj, v_proj, o_proj,
  gate_proj, up_proj, down_proj`.
- `gradient_checkpointing=True` (mandatory on T4).
- `use_cache=False` during training (incompatible with grad checkpointing).
- `per_device_train_batch_size=1` with gradient accumulation 8.
- fp16 compute dtype.

The helper functions (`load_sft_dataset`, `get_lora_target_modules`) come from
`colab_utils.py`, which is uploaded alongside this notebook.


In [ ]:
# Upload colab_utils.py to /content first (or copy from Drive).
# If it lives in your Drive project dir, uncomment the copy below:
# !cp "{PROJECT_DIR}/colab_utils.py" /content/colab_utils.py

import sys
sys.path.insert(0, "/content")
from colab_utils import load_sft_dataset, get_lora_target_modules, print_dataset_summary

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# --- Prepare model for k-bit training + gradient checkpointing ---
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()
# use_cache must be False while gradient checkpointing is on.
model.config.use_cache = False

# --- LoRA configuration ---
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=get_lora_target_modules(),  # q/k/v/o/gate/up/down_proj
)

# --- Load SFT dataset from Drive ---
SFT_DATA_PATH = f"{PROJECT_DIR}/data/sft_train.jsonl"
sft_examples = load_sft_dataset(SFT_DATA_PATH)
print_dataset_summary("SFT", sft_examples)
sft_ds = Dataset.from_list(sft_examples)

# --- SFTTrainer ---
sft_config = SFTConfig(
    output_dir=f"{PROJECT_DIR}/results/sft",
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    gradient_accumulation_steps=SFT_GRADIENT_ACCUMULATION,
    learning_rate=SFT_LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    max_length=MAX_SEQ_LENGTH,
    fp16=True,                       # fp16, NOT bf16 (T4)
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    dataset_text_field=None,         # use chat template via messages field
    packing=False,                   # safer for T4 memory
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=sft_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)

print("Starting SFT training...")
sft_trainer.train()

# --- Save SFT adapter to Drive ---
SFT_ADAPTER_DIR = f"{PROJECT_DIR}/models/sft_adapter"
sft_trainer.model.save_pretrained(SFT_ADAPTER_DIR)
tokenizer.save_pretrained(SFT_ADAPTER_DIR)
print(f"SFT adapter saved to {SFT_ADAPTER_DIR}")


## Cell 6 -- QLoRA DPO Training

Direct Preference Optimization on `dpo_train.jsonl`. We continue from the SFT
adapter saved in Cell 5. The v4 DPO records use
`{"input": {"messages": [...]}, "preferred_output": [...], "non_preferred_output": [...]}`,
which `colab_utils.format_dpo_example` normalizes into the
`{"prompt", "chosen", "rejected"}` conversational form that `DPOTrainer` expects.

DPO uses a smaller learning rate and beta=0.1, with the same T4 memory
constraints (batch size 1, gradient checkpointing, fp16).


In [ ]:
from colab_utils import load_dpo_dataset
from trl import DPOTrainer, DPOConfig
from peft import PeftModel
from datasets import Dataset

DPO_DATA_PATH = f"{PROJECT_DIR}/data/dpo_train.jsonl"

# --- Load DPO dataset ---
dpo_examples = load_dpo_dataset(DPO_DATA_PATH)
print_dataset_summary("DPO", dpo_examples)
dpo_ds = Dataset.from_list(dpo_examples)

# --- Reload base model fresh in 4-bit, then attach the SFT adapter ---
# (Reloading avoids carrying SFT optimizer state into DPO.)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Reloading base model in 4-bit for DPO...")
base_for_dpo = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_for_dpo.config.use_cache = False

# Attach the SFT adapter as the starting point for DPO.
dpo_model = PeftModel.from_pretrained(base_for_dpo, SFT_ADAPTER_DIR, is_trainable=True)
dpo_model.gradient_checkpointing_enable()
dpo_model.config.use_cache = False

dpo_tokenizer = AutoTokenizer.from_pretrained(SFT_ADAPTER_DIR, trust_remote_code=True)
if dpo_tokenizer.pad_token is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

# --- DPO config ---
dpo_config = DPOConfig(
    output_dir=f"{PROJECT_DIR}/results/dpo",
    num_train_epochs=DPO_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=SFT_GRADIENT_ACCUMULATION,
    learning_rate=DPO_LEARNING_RATE,
    beta=DPO_BETA,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_SEQ_LENGTH // 2,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_config,
    train_dataset=dpo_ds,
    processing_class=dpo_tokenizer,
)

print("Starting DPO training...")
dpo_trainer.train()

# --- Save DPO adapter to Drive ---
DPO_ADAPTER_DIR = f"{PROJECT_DIR}/models/dpo_adapter"
dpo_trainer.model.save_pretrained(DPO_ADAPTER_DIR)
dpo_tokenizer.save_pretrained(DPO_ADAPTER_DIR)
print(f"DPO adapter saved to {DPO_ADAPTER_DIR}")


## Cell 7 -- Merge and Save

Merge the DPO LoRA adapter into the base model and save the result in **fp16**
to Drive. We reload the base model in fp16 (not 4-bit) for a clean merge, then
apply the adapter with `merge_and_unload`.

The merged model + tokenizer can then be converted to MLX for local inference
(see `convert_to_mlx.py` in the v4 pipeline).


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MERGED_DIR = f"{PROJECT_DIR}/models/merged"

print("Loading base model in fp16 for clean merge...")
base_merge = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# Attach the DPO adapter then merge it into the base weights.
merged_model = PeftModel.from_pretrained(base_merge, DPO_ADAPTER_DIR)
print("Merging LoRA adapter into base model...")
merged_model = merged_model.merge_and_unload()

# Save merged model (fp16) + tokenizer to Drive.
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
merge_tokenizer = AutoTokenizer.from_pretrained(DPO_ADAPTER_DIR, trust_remote_code=True)
merge_tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")


## Cell 8 -- Quick Smoke Test

Load the merged model and generate a response to a simple prompt. We then run
`colab_utils.evaluate_response` to check that the output:

1. **Begins with a control token** (`[FAST]`, `[THINK]`, `[VERIFY]`,
   `[RECOVER]`, or `[ESCALATE]`).
2. **Uses the tool-call format** (a `<tool_call> ... </tool_call> block
   with valid JSON inside) when a tool is needed.

This is the same structural check the v4 eval suite runs locally.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from colab_utils import smoke_test_prompt, evaluate_response

print("Loading merged model for smoke test...")
smoke_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
smoke_tok = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)

# Build the prompt with the Qwen chat template.
messages = smoke_test_prompt()
input_text = smoke_tok.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = smoke_tok(input_text, return_tensors="pt").to(smoke_model.device)

print("Generating response...")
with torch.no_grad():
    output_ids = smoke_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,        # greedy for reproducibility
        temperature=1.0,
        top_p=1.0,
        pad_token_id=smoke_tok.pad_token_id,
    )

# Decode only the newly generated tokens.
generated = output_ids[0][inputs["input_ids"].shape[1]:]
response = smoke_tok.decode(generated, skip_special_tokens=True)

print("\n=== Generated response ===")
print(response)
print("===========================\n")

# Structural evaluation: control token + tool-call format.
result = evaluate_response(response, require_control_token=True, require_tool_call=True)
print("Evaluation result:")
for k, v in result.items():
    print(f"  {k}: {v}")

if result["passed"]:
    print("\nSMOKE TEST PASSED: control token + tool-call format present.")
else:
    print("\nSMOKE TEST ISSUES:")
    for issue in result["issues"]:
        print(f"  - {issue}")
